Nombre: Dayana Lizeth Robles Moreno

Matrícula: 07320081

Fecha entrega: 25 de septiembre 2026

# TICKET RENOVACALZADO 

In [ ]:

import pdb

import time
# Diccionario con los servicios disponibles de la renovadora
# y sus costos base
servicios = {

    "Cambio De Suela" : 100,
    "Costura" : 300,
    "Tintando" : 500,
    "Restauración" : 900,
    "Ajuste" : 20,
}
# Tiempos de entrega representados como:
# [tiempo base, tiempo extra, límite máximo permitido]
dic_tiempo_entrega = { 
    "Cambio De Suela" : [3, 2, 3 ],
    "Costura" : [5, 3 , 3],
    "Tintando" : [7, 4, 2],
    "Restauración" : [9, 5, 2],
    "Ajuste" : [1, 1, 9]
}
# Archivos de texto que utiliza el sistema para la persistencia de datos
archivos_disponibles = {
    1: "catalogo_servicios.txt",
    2: "politicas_del_negocio.txt",
    3: "promociones_Especiales.txt",
    4: "bitacora_pedidos.txt",
    5: "Base_datos_clientes.txt"
}

# Variables globales para almacenar datos 
#  que serán utilizados en diferentes funciones

fecha_formateada_global = ""
ultimo_cliente = ""
ultimo_telefono = ""

# Recibe 'diccionario_servicios' para poder extraer sus nombres y precios
def mostrar_menu(diccionario_servicios):


    # Impresión encabezado
    print("\n=======MENU========")
    print("========DE=========")
    print("=====SERVICIOS=====\n")
    
    # Guardo las llaves del diccionario para manejar índices fácilmente
    servicios_lista = list(diccionario_servicios.keys())

    #Impresión enumerada de los servicios
    for i, (nombre, costo) in enumerate(diccionario_servicios.items(), start=1):
        print(f" ({i}) {nombre} - ${costo:.2f}")
    
    return servicios_lista

# Recibe un 'mensaje' personalizado para el input, y los límites 'minimo' y 'maximo' 
# Y asegura que no pongan números fuera de rango o letras que rompan el programa.
def pedir_entero(mensaje, minimo, maximo):
    
    # Ciclo infinito hasta que ingrese un valor numérico válido dentro del rango solicitado
    while True:
        try:
            eleccion_cliente = int(input(f"{mensaje} (Rango {minimo}-{maximo}) "))

            # Validación: verifica que esté dentro de los límites permitidos
            if eleccion_cliente >= minimo and eleccion_cliente <= maximo:
                return eleccion_cliente
            else:
                print(f" El valor debe estar entre {minimo} y {maximo}.")
        # Error si meten letras o símbolos en lugar de números
        except ValueError:
            print("Ingresa un número entero válido: solo valor numérico.\n")

# Recibe el nombre del 'servicio' actual, el tiempo acumulado 't_entrega' 
# y la 'cantidad' de veces que se pide para aplicar las reglas de tiempo base y extra.
def calcular_tiempo_entrega(servicio, t_entrega, cantidad):
    
    #Extrae el tiempo base y extra del servicio seleccionado 
    #del dic_tiempo_entrega
    t_base = dic_tiempo_entrega[servicio][0]
    t_extra= dic_tiempo_entrega[servicio][1]
    
    # Regla especial de cálculo de tiempo si el servicio es "Ajuste"
    if servicio == "Ajuste":
         # Mismo día
        if cantidad <= 2:
            t_servicio= 0  
        #Día siguiente
        elif cantidad <= 5:
            t_servicio = t_base
        else:
            t_servicio = t_base + (cantidad - 5) * t_extra
    else:         
        t_servicio = t_base + (cantidad - 1) * t_extra

    # Comparar tiempo actual acumulado del ticket
    if t_servicio > t_entrega:
        t_entrega = t_servicio
        return t_servicio
    else:
        return t_entrega
   
def opcion_cotización():
    # Variables globales que se van a modificar o consultar
    global ultimo_cliente, ultimo_telefono, fecha_formateada_global

    # Muestra el menú de servicios y guarda la lista de opciones
    servicios_opciones = mostrar_menu(servicios)

    # Datos generales del cliente capitalizando su nombre
    cliente = input("Ingrese nombre completo del cliente: ").capitalize()
    telefono = input("Ingrese número telefónico del cliente: ")
    
    #Estas funcionaran para modificar manualmente el archivo de base de datos.
    ultimo_cliente = cliente
    ultimo_telefono = telefono

    # Pide la cantidad total de servicios que desea cotizar
    c_pedidos = pedir_entero("Ingrese cantidad de servicios totales.", 1, 10)
    

    # Diseño del ticket.

    print("\n=======================")
    print("=========TICKET==========")
    print("======RENOVACALZADO======")
    print("=========================")
    print(f"Nombre cliente: {cliente}")

    # Va a ir guardando los servicios solicitados del cliente.
    # su función es poder manejar las limitaciones de cantidad por sevicio.
    ticket_servicios=[]

    subtotal=0
    t_entrega = 0
    m_descuento = 0

    # Iteración para capturar cada servicio de forma individual
    for pedido in range(1, c_pedidos +1):
        
        while True:
            servicio_elegido = pedir_entero(f"Ingrese Servicio{pedido}/{c_pedidos}: ", 1, 5)
            servicio_real = servicios_opciones[servicio_elegido-1]
            limite = dic_tiempo_entrega[f"{servicio_real}"][2]
            intentos_actuales = ticket_servicios.count(servicio_real) + 1

            # Valida si no sobrepasa el límite permitido para ese servicio
            if ticket_servicios.count(servicio_real) < limite:
                print(f"Servicio {servicio_real}: {intentos_actuales} de Máximo {limite}")
                break
            else:
                print(f"Servicio {servicio_real}: Supero el límite.{ticket_servicios.count(servicio_real)}/{limite} ")

        # Agrego el servicio después de validar
        ticket_servicios.append(servicio_real)

        # Sumo el costo del servicio actual al subtotal
        subtotal += servicios.get(servicio_real)
        
        # Calcula y actualiza el tiempo de entrega según la cantidad de días mayores
        cantidad_servicio = ticket_servicios.count(servicio_real)
        t_entrega = calcular_tiempo_entrega(servicio_real, t_entrega, cantidad_servicio)

    print(f"\nSubtotal acumulado: ${subtotal:.2f}")
    ##-----Termina captura de servicios de manera independiente----

    ##-----Inician cálculos para ticket: descuento, tiempo de entrga final, servicio express----
    
    # Aplico descuento del 10% si los servicios totales son mayores a 2
    if c_pedidos > 2:
       print("Descuento: Sí aplica")
       m_descuento = subtotal * 0.10
    else:
       m_descuento = 0

    # Preguntar por servicio express si el tiempo de entrega es mayor a un día
    m_express = 0
    if t_entrega > 1:
        while True:
            urgente = input("\n¿Servicio Express? (Ingrese 'si' o 'no'): ").strip().lower()
            if urgente == "si" or urgente == "no":
                break
            print("Respuesta no válida. Ingrese 'si' o 'no'.")

        if urgente == "si":
            m_express = subtotal * 0.20
            t_entrega = t_entrega // 2
            print("Servicio Express aplicado: +20%")
        else:
            print("Servicio Express no solicitado.")
    else:
        print("El pedido no aplica para Servicio Express (entrega rápida o mismo día).")

    # Calcula el total definitivo de la cotización
    total = subtotal - m_descuento + m_express

    if m_descuento > 0:
        print(f"Monto descuento: ${m_descuento:.2f}")
    if m_express > 0:
        print(f"Monto Servicio Express: ${m_express:.2f}")

    print(f"\nTOTAL A PAGAR: ${total:.2f}")

    # Muestra tiempos de entrega finales
    if t_entrega == 0:
         print("Tiempo de entrega: mismo día.")
    else:
        print(f"Tiempo estimado de entrega: {t_entrega} días después de anticipo.")

    # Anticipo mínimo 50%, SÍ puede dejar liquidado
    anti_min = total * 0.50
    while True:
        try:
            anticipo = float(input(f"Ingrese el anticipo (Mínimo 50%, equivale a ${anti_min:.2f}): $"))
            if anti_min <= anticipo <= total:
                break
            elif anticipo > total:
                print(f"El anticipo no puede ser mayor al total: ${total:.2f}")
            else:
                print(f"El anticipo debe ser al menos del 50%: ${anti_min:.2f}")
        except ValueError:
            print("Ingrese un valor numérico válido para el anticipo.")

    s_pendiente = total - anticipo
    print(f"Anticipo pagado: ${anticipo:.2f}")

    if anticipo == total:
        print("Pedido liquidado.")
    else:
        print(f"Restante a pagar: ${s_pendiente:.2f}")

    print("\n========¡Buen día!========\nGracias por elegir 'RENOVACALZADO'")

    #Guardado automático en bitacora_pedidos,txt del ticket generado
    try:
        detalle_ticket = f"Fecha: {fecha_formateada_global} | Cliente: {cliente} | Total: ${total:.2f}\n"

        with open("bitacora_pedidos.txt", "a") as archivo_bitacora:
            archivo_bitacora.write(detalle_ticket)
        print("[Sistema]: Ticket guardado exitosamente en 'bitacora_pedidos.txt'.")
    
    except Exception as e:
        print(f"Error: {e}")

# función de tiempo de carga visual al entrar al sistema
def pantalla_carga():
    print("\nIniciando sistema...\n")
   
    for i in range(1, 6):
        print(f"Cargando módulos... [{i}/5s]")
        time.sleep(1)
    print("\n¡Carga completada con éxito!\n")

# Función que pide identificación al usuario o cerrar todo el sistema
def login():
    print("\nIngrese usuario (o escriba 'Salir' para cerrar el sistema). ")
    usuario = input("").capitalize()
    
    if usuario == "Salir":
        print("Cerrando el sistema. ¡Hasta luego!")
        return False
    
    print(f"\nBienvenido al sistema, {usuario} ")
    return True

def capturar_fecha():
    bisiesto = False
    anio = pedir_entero("Ingrese año: ", 2020, 2026)
    # Comprueba si el año ingresado es bisiesto 
    if (anio % 4 == 0 and anio % 100 != 0) or (anio % 400 == 0):
        bisiesto = True
    mes = pedir_entero("Ingrese mes: ", 1, 12)
    # Valida los días máximos permitidos según el mes y si es bisiesto
    if mes == 2 and bisiesto:
        dia = pedir_entero("Ingrese día: ", 1, 29)
    elif mes == 2 and not bisiesto:
        dia = pedir_entero("Ingrese día: ", 1, 28)
    elif mes in [4, 6, 9, 11]:
        dia = pedir_entero("Ingrese día: ", 1, 30)
    else:
        dia = pedir_entero("Ingrese día: ", 1, 31)


    fecha = (dia, mes, anio)
    return fecha

#Función de simulación en caso de que el usuario esté inactivo por 10 minutos.
def inactividad():
    tiempo_espera= 600
    for segundo in range(tiempo_espera):
        time.sleep(1)

    print("\n--- AVISO DE INACTIVIDAD ---")
    print("Han transcurrido 10 minutos sin interacción u opción seleccionada.")
    
    while True:
        respuesta_usuario = input('¿Desea continuar? Escriba "si" o "no": ').strip().lower()
        
        if respuesta_usuario == "si":
            return True 
        elif respuesta_usuario == "no":
            return False  
        else:
            print('Debe escribir "si" o "no".')

# Función que se encarga de mostrar los archivos disponibles para leer.
def leer_archivo():

    print("\n--- Leer Archivos ---")
    print("Archivos disponibles en el sistema:")

    for clave, nombre in archivos_disponibles.items():
        print(f" [{clave}] {nombre}")

    # Valida que sea un archivo dentro del diccionario
    # lee el archivo elegido mediante un bloque try-except y evita que el programa caiga si no existe.
    while True:
        try:
            eleccion = int(input("\nIngrese el número del archivo que desea abrir: "))
            
            if eleccion in archivos_disponibles:
                nombre_archivo = archivos_disponibles[eleccion]

                # Abre el archivo y muestra su contenido
                with open(nombre_archivo, "r") as archivo:
                    # guarda el contenido
                    contenido = archivo.read()
                    print(f"\n--- Contenido de '{nombre_archivo}' ---")
                    if contenido.strip() == "":
                        print("(El archivo está actualmente vacío)")
                    else:
                        print(contenido)
                    print("-------------------FIN ARCHIVO--------------------")
                    break
            else:
                print("Opción de archivo no válida. Elija un número entre 1 y 5.")
            
        except ValueError:
            print("Error: Debe ingresar un valor numérico entero válido.")
        except FileNotFoundError:
            print("Error: El archivo seleccionado no existe en el sistema.")



# No recibe parámetros. Maneja la lógica para escribir o anexar datos a los documentos 
# protegidos por restricciones (como el catálogo) y permite usar datos manuales o del último ticket.
def escribir_archivo():

    global fecha_formateada_global, ultimo_cliente, ultimo_telefono
    print("\n---- Escribir o Editar Archivo ---")
    
    print("Seleccione el archivo que desea modificar: ")
    for clave, nombre in archivos_disponibles.items():
        print(f" [{clave}] {nombre}")
        
 
    eleccion = pedir_entero("Ingrese el número del archivo", 1, 5)
    
    # Restricción del archivo 1 (Catálogo) Evitar que agregen cosas al archivo
    if eleccion == 1:
        print("[Acceso Denegado]: No está permitido modificar ni alterar el Catálogo de Servicios.")
        return
        
    if eleccion in archivos_disponibles:
        nombre_archivo = archivos_disponibles[eleccion]

        
        #Si elige base de datos: puede guardar datos manuales o del último ticket.
        if eleccion == 5:
            print("\n--- Base de Datos de Clientes ---")
            print(" (1) Ingreso manual (escribir nombre y teléfono libremente)")
            print(" (2) Usar datos del último ticket generado en el programa")
            origen_datos = pedir_entero("Elija una opción", 1, 2)
            
            if origen_datos == 1:
                nombre_manual = input("Ingrese el nombre del cliente: ")
                tel_manual = input("Ingrese el teléfono del cliente: ")
                texto_a_guardar = f"Cliente: {nombre_manual} | Teléfono: {tel_manual}"
            else:
                if ultimo_cliente == "":
                    print("[Aviso]: Aún no se ha generado ninguna cotización en esta sesión para extraer datos.")
                    return
                texto_a_guardar = f"Cliente: {ultimo_cliente} | Teléfono: {ultimo_telefono}"
        else:
            # Para políticas, promos o bitácora pide el texto directamente
            texto_a_guardar = input("Ingrese la información que desea guardar: ")
        
        print("\n¿Qué acción deseas realizar en el archivo?")
        print(" (1) Anexar información (agregar al final)")
        print(" (2) Reescribir archivo (borrar anterior y poner nuevo)")
        
        accion = pedir_entero("Ingrese opcion", 1, 2)
        modo = "a" if accion == 1 else "w"
        # Escribe en el archivo integrando la fecha global y el texto configurado
        try:
            with open(nombre_archivo, modo) as archivo:
                archivo.write(f"\nFecha: {fecha_formateada_global}: {texto_a_guardar}\n")
                print(f"¡Datos guardados con éxito en '{nombre_archivo}'!")
        
        except FileNotFoundError:
            print("Error: El archivo seleccionado no existe.")

# Función principal del programa. No recibe parámetros.
def main(): 
    
    global fecha_formateada_global
    # Bucle al que regresa cerrando sesión o 
    # Después de un periodo de inactividad y selecciona que ya no quiere continuar 
    while True:
        print("\n========================")
        print("\n SISTEMA RENOVACALZADO ")
        print("\n========================")
        
        #Llama a la función Login(), aquí elige: identificarse o salir de todo el progama
        if not login():
            return

        # Inicia la pantalla de carga una vez identificandose
        pantalla_carga()

        
        print("--- Configuración inicial de Fecha ---")
        tupla_recibida = capturar_fecha()
        fecha_formateada_global = f"{tupla_recibida[0]:02d}/{tupla_recibida[1]:02d}/{tupla_recibida[2]:04d}"
        print(f"Fecha: {fecha_formateada_global}")
        
        sesion_activa=True

        
        while sesion_activa:
            # Encabezado
            print("\n=====================")
            print("\n===Menú Principal====")
            print("\n=====================")

            # Matriz para imprimir las opciones del menú de forma organizada
            matriz_menu_principal = [
                    ["(1) Cotizar Servicio", "(2) Leer Archivos"],
                    ["(3) Escribir/anexar documentos", "(4) Cerrar Usuario"]
                ]
            
            for fila in matriz_menu_principal:
                for columna in fila:
                    print(columna)

            print("\n[Iniciando simulación por inactividad]")
            continuar = inactividad()
            
            if not continuar:
                print("\n[Sesión suspendida por inactividad]. Regresando al inicio...")
                sesion_activa = False  
                break     

            opcion_menu = pedir_entero("¿Qué desea hacer? Eliga una opción: ", 1, 4)
        
            # Opción cotizar
            if opcion_menu == 1:
                print("\n[Opción 1 seleccionada: Cotización de servicio]")
                opcion_cotización()

            # Opción leer archivos
            elif opcion_menu == 2:
                print("\n[Opción 2 seleccionada: Leer archivo de texto]")
                leer_archivo()

            # Opción escirbir archivos
            elif opcion_menu == 3:
                print("\n[Opción 3 seleccionada: Escribir o anexar datos]")
                escribir_archivo()

            # Cerrar sesión
            elif opcion_menu == 4:
                print("Cerrando sesión. ¡Hasta luego!")
                break


main()




 SISTEMA RENOVACALZADO 


Ingrese usuario (o escriba 'Salir' para cerrar el sistema). 

Bienvenido al sistema,  

Iniciando sistema...

Cargando módulos... [1/5s]
Cargando módulos... [2/5s]
Cargando módulos... [3/5s]
Cargando módulos... [4/5s]
Cargando módulos... [5/5s]

¡Carga completada con éxito!

--- Configuración inicial de Fecha ---
Ingresa un número entero válido: solo valor numérico.

Fecha: 25/02/2026


===Menú Principal====

(1) Cotizar Servicio
(2) Leer Archivos
(3) Escribir/anexar documentos
(4) Cerrar Usuario

[Iniciando simulación por inactividad]

--- AVISO DE INACTIVIDAD ---
Han transcurrido 10 minutos sin interacción u opción seleccionada.
Debe escribir "si" o "no".

[Opción 1 seleccionada: Cotización de servicio]

=======MENU========
========DE=========
=====SERVICIOS=====

 (1) Cambio De Suela - $100.00
 (2) Costura - $300.00
 (3) Tintando - $500.00
 (4) Restauración - $900.00
 (5) Ajuste - $20.00

=========TICKET==========
======RENOVACALZADO======
Nombre cliente: